In [0]:
use databricks_wanderbricks_dataset_dais_2025.wanderbricks;
show tables;
select * from property_amenities limit 100;
select * from amenities ;

select count(*) from amenities;


-- 1. total amentities.
Select count(amenity_id) AS total_amenities
from amenities;

-- 2. What amentity category exist.
Select distinct category
from amenities;

--- 3 most common amenites
SELECT 
    a.name AS amenity,
    COUNT(pa.amenity_id) AS usage_count
FROM property_amenities pa
JOIN amenities a
    ON pa.amenity_id = a.amenity_id
GROUP BY a.name
ORDER BY usage_count DESC;

--- 4 How many amenities does each property have 
select property_id, COUNT(amenity_id) AS total_amenities
from property_amenities
group by property_id
order by total_amenities desc;

--- 5. avg amenities per property 
Select AVG(amenity_count) AS avg_amenities_per_property
from (
select property_id, count(distinct amenity_id) AS amenity_count
    from property_amenities
    group by property_id
) AS property_amenities_count;

--- 6. most common amenities within each country

select c.country,a.name as amenity,count(*) as amenity_count,
row_number()over(partition by c.country order by count(*) desc)as rank_no
from property_amenities pa
join amenities a
on pa.amenity_id=a.amenity_id
join properties p
on pa.property_id=p.property_id
join cities c
on p.city_id=c.city_id
group by c.country,a.name
qualify rank_no<=3
order by c.country,amenity_count desc;



--- 7. avg property price by amenity category

select 
case 
when total_properties >= 10 then '10+ Properties' 
when total_properties between 7 and 9 then '7-9 Properties' 
when total_properties between 4 and 6 then '4-6 Properties' 
when total_properties between 2 and 3 then '2-3 Properties' 
else '1 Property' 
end as property_range, 
count(*) as total_hosts 
from ( select h.host_id, count(*) as total_properties from hosts h join properties p on h.host_id = p.host_id where h.is_active = true group by h.host_id ) 
group by property_range 
order by case when property_range = '10+ Properties' then 1 
when property_range = '7-9 Properties' then 2 
when property_range = '4-6 Properties' then 3 
when property_range = '2-3 Properties' then 4 
else 5 end;

